# 04 – Pipeline-Oberfläche

Eine Gradio-Oberfläche für alle drei Stufen und den Arbeitsbereich:

| Reiter | Inhalt |
|---|---|
| **Auftrag** | PDFs hochladen → *Neuer Auftrag* leert `data/`, legt die Struktur an und übernimmt die PDFs nach `data/raw`. Status je Dokument. |
| **Lauf** | Dokumente und Stufen wählen (1 Layout+OCR+Docling · 2 Lektorat · 3 Bilder), Protokoll läuft live mit. Optional *danach ernten + zurücksetzen*. |
| **Ergebnis** | Vorschau (lektoriert / OCR-Rohfassung), Bilder mit Alt-Text, Lektorat-Review. *Ernten + zurücksetzen*. |
| **Einstellungen** | Je Stufe Anbieter und Modell: LM Studio (lokal) oder **DeepInfra** für Lektorat und Bilder, gespeichert in `pipeline_config.json`. Der DeepInfra-API-Key gilt nur für die laufende Sitzung (nie auf der Platte; ersatzweise Umgebungsvariable `DEEPINFRA_API_KEY`), Reasoning wird automatisch abgeschaltet. |

**Ernte:** `<BUCH>_korr.md`, `<BUCH>.json` und `<BUCH>_artifacts/` wandern nach
`ergebnisse/<BUCH>_<Datum_Uhrzeit>/`. Zurückgesetzt wird erst, wenn *jede* Ernte gelang –
bricht ein Lauf ab, bleiben Befunde und Checkpoints liegen, und der nächste Lauf setzt dort auf.

Die Oberfläche öffnet sich in einem eigenen Browserfenster. Solange sie läuft, ist dieser Kernel belegt;
**Kernel → Interrupt** (oder die Zelle unten) beendet sie.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (cwd, *cwd.parents) if (p / "python" / "pipeline").is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Projekt-Root nicht gefunden: python/pipeline fehlt.")
if str(PROJECT_ROOT / "python") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "python"))

%load_ext autoreload
%autoreload 2

from pipeline import arbeitsbereich, gui
from pipeline.konfig import PipelineKonfig, KONFIG_DATEI

print("Projekt   :", PROJECT_ROOT)
print("Konfig    :", KONFIG_DATEI, "(vorhanden)" if KONFIG_DATEI.exists() else "(Standardwerte)")
for s in arbeitsbereich.status():
    print("  ", s.zeile())

In [ ]:
# Oberfläche starten. prevent_thread_lock=True gibt den Kernel sofort wieder frei;
# die Oberfläche läuft weiter, bis demo.close() oder der Kernel neu startet.
demo = gui.baue_oberflaeche()
konfig = PipelineKonfig.laden()
demo.queue(default_concurrency_limit=4)
demo.launch(inbrowser=True, prevent_thread_lock=True,
            allowed_paths=[str(PROJECT_ROOT), str(konfig.ergebnis_pfad)],
            css=gui.CSS, theme=gui.gr.themes.Soft())

In [ ]:
# Oberfläche beenden.
demo.close()

---
## Ohne Oberfläche: dieselben Aufrufe direkt

```python
from pipeline import steuerung
k = PipelineKonfig.laden()
arbeitsbereich.neuer_auftrag([Path("~/Downloads/Buch.pdf").expanduser()])
steuerung.gesamtlauf(k, stufen=(1, 2, 3), ernten=True)
```